In [7]:
# Cell 1 — setup Drive, environment, and local HuggingFace cache

!pip install -q transformers accelerate faiss-cpu sentencepiece

import os
import glob
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import faiss

import os
from google.colab import drive

# Start from /content so we are not stuck in a broken gdrive path
os.chdir("/content")
drive.mount("/content/gdrive", force_remount=True)

# Project path
PROJECT_DIR = "/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj" #VANSH'S PATH
#PROJECT_DIR = "/content/gdrive/MyDrive/Colab Notebooks/final-proj"

# Important environment variables
os.environ["FAISS_NO_AVX2"] = "1"
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"

# Keep HuggingFace cache local, not in Drive
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"

!mkdir -p /content/hf_cache/datasets /content/hf_cache/transformers

# Move into project folder
os.chdir(PROJECT_DIR)
print("Now working in:", os.getcwd())


ValueError: Mountpoint must not already contain files

In [8]:
import os
os.chdir("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")
print(os.getcwd())

/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj


In [9]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([
    ["ColBERT strict Recall@5 on NQ", "32.78%", ""],
    ["ColBERT + generic BART, K=5", "0.00 EM", "0.07 F1"],
    ["ColBERT + fine-tuned NQ BART, K=5", "0.22 EM", "0.27 F1"],
    ["ColBERT + adapted BART 5k, K=5", "0.20 EM", "0.26 F1"],
    ["ColBERT + adapted BART 5k, K=50", "0.18 EM", "0.23 F1"],
], columns=["Experiment", "Primary Result", "Secondary Result"])

out_path = RESULTS_DIR / "colbert_experiment_summary.csv"
summary.to_csv(out_path, index=False)

print("Saved:", out_path)
summary

Saved: /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/results/colbert_experiment_summary.csv


,Experiment,Primary Result,Secondary Result
0,ColBERT strict Recall@5 on NQ,32.78%,
1,"ColBERT + generic BART, K=5",0.00 EM,0.07 F1
2,"ColBERT + fine-tuned NQ BART, K=5",0.22 EM,0.27 F1
3,"ColBERT + adapted BART 5k, K=5",0.20 EM,0.26 F1
4,"ColBERT + adapted BART 5k, K=50",0.18 EM,0.23 F1


In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"


import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

torch: 2.9.0+cu128
CUDA: True


In [ ]:
#!pip install torchvision==0.24.0 torchaudio==2.9.0
#Run the above by ITSELF first
#then you restart runtime, comment out the above line and run the bottom
# it'll make you restart runtime once more and then you're chilling
!pip install pylate

In [ ]:
from pylate import indexes, models, retrieve
print("Imported")

Imported


In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")

if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj")

os.chdir(PROJECT_DIR)

print("Now in:", os.getcwd())
!ls -lh data/small_nq_index
!find data/small_nq_index -maxdepth 2 -type f -print

Now in: /content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj
total 147M
-rw------- 1 root root 147M May  4 12:47 index.faiss
drwx------ 2 root root 4.0K May  4 12:47 passages
data/small_nq_index/index.faiss
data/small_nq_index/passages/data-00000-of-00001.arrow
data/small_nq_index/passages/state.json
data/small_nq_index/passages/dataset_info.json


In [ ]:
# import os
# import pandas as pd
# from huggingface_hub import hf_hub_download
# from pylate import indexes, models

# print("GPU:", torch.cuda.get_device_name(0))

# print("loading passages...\n")
# shard_files = []
# for i in range(4):
#     f = hf_hub_download(
#         repo_id="facebook/wiki_dpr",
#         filename=f"data/psgs_w100/nq/train-{i:05d}-of-00157.parquet",
#         repo_type="dataset",
#         cache_dir="/content/hf_cache",
#     )
#     shard_files.append(f)

# dfs = [pd.read_parquet(f) for f in shard_files]
# passages_df = pd.concat(dfs, ignore_index=True)
# print("Columns in passages:\n ", passages_df.columns)
# # Take exactly 50K
# SUBSET_SIZE = 50_000
# passages_sample = passages_df.sample(n=SUBSET_SIZE, random_state=42).reset_index(drop=True)
# documents = (passages_sample["title"] + ". " + passages_sample["text"]).tolist()
# doc_ids = [str(i) for i in range(len(passages_sample["id"].tolist()))]

# print("Now loading colbertv2 model")
# model = models.ColBERT(model_name_or_path='lightonai/colbertv2.0', device='cuda')
# print(f"\nEncoding {len(documents):,} documents...")
# doc_embeddings = model.encode(
#     documents,
#     batch_size=64,
#     is_query=False,
#     show_progress_bar=True,
# )

# # Build the index
# print("\nBuilding PLAID index...")
# index = indexes.PLAID(
#     index_folder="/content/pylate_index",
#     index_name="wiki_colbert_50k",
#     override=True,
# )
# index.add_documents(documents_ids=doc_ids, documents_embeddings=doc_embeddings)
# print("\nDone.")

# Load the exact same 50k passage corpus used by the old NQ RAG index

from datasets import load_from_disk
import pandas as pd
from pathlib import Path

SMALL_INDEX_DIR = Path.cwd() / "data/small_nq_index"
PASSAGES_DIR = SMALL_INDEX_DIR / "passages"
FAISS_INDEX_PATH = SMALL_INDEX_DIR / "index.faiss"

print("SMALL_INDEX_DIR:", SMALL_INDEX_DIR)
print("PASSAGES_DIR:", PASSAGES_DIR)
print("FAISS_INDEX_PATH:", FAISS_INDEX_PATH)

assert PASSAGES_DIR.exists(), f"Missing {PASSAGES_DIR}"
assert FAISS_INDEX_PATH.exists(), f"Missing {FAISS_INDEX_PATH}"

passages_ds = load_from_disk(str(PASSAGES_DIR))
passages_sample = pd.DataFrame(passages_ds)

print("Loaded passages:", len(passages_sample))
print(passages_sample.columns)
print(passages_sample.head())

SMALL_INDEX_DIR: /content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj/data/small_nq_index
PASSAGES_DIR: /content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj/data/small_nq_index/passages
FAISS_INDEX_PATH: /content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj/data/small_nq_index/index.faiss
Loaded passages: 50000
Index(['title', 'text', 'embeddings'], dtype='object')
                title                                               text  \
0  NQ train example 0  Question: where did they film hot tub time mac...   
1  NQ train example 1  Question: who has the right of way in internat...   
2  NQ train example 2  Question: who does annie work for attack on ti...   
3  NQ train example 3  Question: when was the immigration reform and ...   
4  NQ train example 4  Question: when was puerto rico added to the us...   

                                          embeddings  
0  [0.2267296314239502, -

In [ ]:
# Build ColBERT index over the SAME 50k corpus

from pylate import indexes, models, retrieve
import torch

COLBERT_INDEX_FOLDER = "/content/pylate_nq_small_index"
COLBERT_INDEX_NAME = "small_nq_colbert_50k"

documents = (
    passages_sample["title"].fillna("").astype(str)
    + ". "
    + passages_sample["text"].fillna("").astype(str)
).tolist()

doc_ids = [str(i) for i in range(len(documents))]

print("Loading ColBERT model...")
colbert_model = models.ColBERT(
    model_name_or_path="lightonai/colbertv2.0",
    device="cuda",
)

print(f"Encoding {len(documents):,} documents...")
doc_embeddings = colbert_model.encode(
    documents,
    batch_size=64,
    is_query=False,
    show_progress_bar=True,
)

print("Building PLAID index...")
colbert_index = indexes.PLAID(
    index_folder=COLBERT_INDEX_FOLDER,
    index_name=COLBERT_INDEX_NAME,
    override=True,
)

colbert_index.add_documents(
    documents_ids=doc_ids,
    documents_embeddings=doc_embeddings,
)

colbert_retriever = retrieve.ColBERT(index=colbert_index)

print("ColBERT index built on old small_nq_index corpus.")

Loading ColBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/669 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/582 [00:00<?, ?B/s]

Encoding 50,000 documents...


Encoding documents (bs=64):   0%|          | 0/782 [00:00<?, ?it/s]

Building PLAID index...
Computing centroids of embeddings.
Creating FastPlaid index.
ColBERT index built on old small_nq_index corpus.


In [ ]:
def colbert_fn(question, k=5):
    q_emb = colbert_model.encode(
        [question],
        batch_size=1,
        is_query=True,
        show_progress_bar=False,
    )

    results = colbert_retriever.retrieve(
        queries_embeddings=q_emb,
        k=k,
    )

    docs = []
    for r in results[0]:
        row = int(str(r["id"]))
        p = passages_sample.iloc[row]

        docs.append({
            "row": row,
            "score": float(r.get("score", 0.0)),
            "title": str(p["title"]),
            "text": str(p["text"]),
        })

    return docs

In [ ]:
#Building Colbert-NQ training data

from pathlib import Path
import json
from tqdm.auto import tqdm
from eval_harness import load_dataset

PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj")

TRAIN_PATH = PROJECT_DIR / "data/nq_train.jsonl"

ADAPT_DIR = PROJECT_DIR / "outputs/colbert_bart_adapted_generator"
ADAPT_DIR.mkdir(parents=True, exist_ok=True)

ADAPT_TRAIN_PATH = ADAPT_DIR / "nq_train_colbert_contexts_5k.jsonl"

MAX_TRAIN = 5000
K = 5

nq_train = load_dataset(str(TRAIN_PATH), max_examples=MAX_TRAIN)
print(f"Loaded {len(nq_train)} NQ train examples")

def format_colbert_input(question, docs):
    parts = [f"question: {question}"]
    for i, d in enumerate(docs, start=1):
        title = str(d.get("title", ""))
        text = str(d.get("text", str(d)))
        parts.append(f"document {i}: {title}. {text[:700]}")
    return "\n".join(parts)

if ADAPT_TRAIN_PATH.exists():
    print("Training file already exists:", ADAPT_TRAIN_PATH)
else:
    with open(ADAPT_TRAIN_PATH, "w", encoding="utf-8") as f:
        for ex in tqdm(nq_train, desc="Retrieving ColBERT contexts"):
            q = ex["question"]
            answers = ex.get("answers") or ex.get("all_answers") or []

            if not answers:
                continue

            docs = colbert_fn(q, k=K)
            source = format_colbert_input(q, docs)
            target = str(answers[0])

            row = {
                "question": q,
                "source": source,
                "target": target,
                "answers": answers,
            }

            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Saved:", ADAPT_TRAIN_PATH)

ModuleNotFoundError: No module named 'eval_harness'

In [ ]:
#fine-tune bart on colbert contexts

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import json
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    train_bart_model = nq_bart_model
    train_bart_tokenizer = nq_bart_tokenizer
    print("Using fine-tuned NQ RAG BART generator")
except NameError:
    train_bart_model = bart_model
    train_bart_tokenizer = bart_tokenizer
    print("Using current bart_model / bart_tokenizer")

train_bart_model = train_bart_model.to(DEVICE)
train_bart_model.train()

PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj")

ADAPT_DIR = PROJECT_DIR / "outputs/colbert_bart_adapted_generator"
ADAPT_TRAIN_PATH = ADAPT_DIR / "nq_train_colbert_contexts_5k.jsonl"
SAVE_DIR = ADAPT_DIR / "bart_colbert_nq_5k"

class ColBERTBartDataset(Dataset):
    def __init__(self, path):
        self.rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.rows.append(json.loads(line))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

def collate_fn(batch):
    sources = [b["source"] for b in batch]
    targets = [b["target"] for b in batch]

    enc = train_bart_tokenizer(
        sources,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=768,
    )

    dec = train_bart_tokenizer(
        targets,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=32,
    )

    labels = dec["input_ids"]
    labels[labels == train_bart_tokenizer.pad_token_id] = -100

    enc["labels"] = labels
    return enc

dataset = ColBERTBartDataset(ADAPT_TRAIN_PATH)
loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

EPOCHS = 1
LR = 2e-5
GRAD_ACCUM = 8

optimizer = AdamW(train_bart_model.parameters(), lr=LR)
total_steps = (len(loader) // GRAD_ACCUM + 1) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(10, total_steps // 10),
    num_training_steps=total_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

global_step = 0
optimizer.zero_grad()

for epoch in range(EPOCHS):
    pbar = tqdm(loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for step, batch in enumerate(pbar):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            outputs = train_bart_model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            pbar.set_postfix({
                "loss": float(loss.item() * GRAD_ACCUM),
                "step": global_step,
            })

print("Saving adapted BART generator to:", SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_bart_model.save_pretrained(SAVE_DIR)
train_bart_tokenizer.save_pretrained(SAVE_DIR)

print("Done.")

NameError: name 'bart_model' is not defined

In [ ]:
#replace bart_fn with adapted BART

import torch
from pathlib import Path
from transformers import BartForConditionalGeneration, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj")

SAVE_DIR = PROJECT_DIR / "outputs/colbert_bart_adapted_generator/bart_colbert_nq_5k"

adapted_bart_model = BartForConditionalGeneration.from_pretrained(SAVE_DIR).to(DEVICE)
adapted_bart_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

adapted_bart_model.eval()

def format_colbert_input(question, docs):
    parts = [f"question: {question}"]
    for i, d in enumerate(docs, start=1):
        title = str(d.get("title", ""))
        text = str(d.get("text", str(d)))
        parts.append(f"document {i}: {title}. {text[:700]}")
    return "\n".join(parts)

@torch.no_grad()
def bart_fn(question, docs, max_new_tokens=16, num_beams=4):
    source = format_colbert_input(question, docs)

    enc = adapted_bart_tokenizer(
        source,
        return_tensors="pt",
        truncation=True,
        max_length=768,
    ).to(DEVICE)

    out = adapted_bart_model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

    return adapted_bart_tokenizer.decode(out[0], skip_special_tokens=True).strip()

print("Replaced bart_fn with ColBERT-adapted BART generator.")

Loading weights:   0%|          | 0/514 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Replaced bart_fn with ColBERT-adapted BART generator.


In [ ]:
import numpy as np
import faiss

embeddings = np.vstack(passages_sample['embeddings'].values).astype(np.float32)
print(f"Shape: {embeddings.shape}")

print("Now building index...")
dpr_index = faiss.IndexFlatIP(768)
dpr_index.add(embeddings)

#faiss.write_index(dpr_index, "/content/gdrive/MyDrive/Colab Notebooks/final-proj/dpr_50k.faiss")
faiss.write_index(dpr_index, "/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/dpr_50k.faiss")


#"/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/dpr_50k.faiss"

Shape: (50000, 768)
Now building index...


In [ ]:
##Loading BART
import torch
from transformers import BartForConditionalGeneration, AutoTokenizer

print("Loading BART (vblagoje/bart_lfqa)...")
bart_model = BartForConditionalGeneration.from_pretrained(
    "vblagoje/bart_lfqa"
).to("cuda").eval()
bart_tokenizer = AutoTokenizer.from_pretrained("vblagoje/bart_lfqa")
print("Done.")

Loading BART (vblagoje/bart_lfqa)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Done.


In [ ]:
import torch
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer

print("Loading DPR question encoder...")
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base").to("cuda").eval()

@torch.no_grad()
def dpr_search(question, k=5):
    inputs = q_tokenizer(question, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    q_emb = q_encoder(**inputs).pooler_output.cpu().numpy().astype(np.float32)
    scores, ids = dpr_index.search(q_emb, k)
    results = []
    for s, i in zip(scores[0], ids[0]):
        p = passages_sample.iloc[int(i)]
        results.append((float(s), p["title"], p["text"]))
    return results

# Test
for q in ["What is the capital of France?", "Who wrote Pride and Prejudice?", "When did World War II end?"]:
    print(f"\nQ: {q}")
    for score, title, text in dpr_search(q, k=3):
        print(f"  [{score:.2f}] {title}: {text[:120]}...")

Loading DPR question encoder...


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]


Q: What is the capital of France?
  [71.38] Guangzhou: collection of ancient books in Southern China. Guangzhou currently maintains sister city agreements with the following f...
  [71.29] Macau: Macau Macau or Macao (; , ; ), officially the Macao Special Administrative Region of the People's Republic of China, is ...
  [70.65] Kathmandu: Kathmandu Kathmandu (; , "Yei", Nepali pronunciation: ) is the capital city and largest city of Nepal with a population ...

Q: Who wrote Pride and Prejudice?
  [74.94] "Kōan": Kōan A (; , ; "gong-an"; ) is a story, dialogue, question, or statement which is used in Zen practice to provoke the "gr...
  [73.34] "Li Bai": including another Taoist (and poet), He Zhizhang, who bestowed upon him the nickname the "Immortal Exiled from Heaven". ...
  [73.06] "Kōan": on the use of koans. He saw the kung-ans as "work of literature [that] should be used as objective, universal standards ...

Q: When did World War II end?
  [66.50] "Hainish Cycle": destroyed b

In [ ]:
# Build id→row lookup if your ColBERT doc_ids are passage IDs (not row indices)
id_to_row = {str(pid): i for i, pid in enumerate(passages_sample["id"])}
colbert_retriever = retrieve.ColBERT(index=index)

def colbert_search(question, k=5):
    q_emb = model.encode([question], batch_size=1, is_query=True, show_progress_bar=False)
    results = colbert_retriever.retrieve(queries_embeddings=q_emb, k=k)
    out = []
    for r in results[0]:
        if r["id"] in id_to_row:
            row = id_to_row[r["id"]]
        else:
            row = int(r["id"])
        p = passages_sample.iloc[row]
        out.append((r["score"], p["title"], p["text"]))
    return out

for q in ["What is the capital of France?", "Who wrote Pride and Prejudice?", "When did World War II end?"]:
    print(f"\nQ: {q}")
    for score, title, text in colbert_search(q, k=3):
        print(f"  [{score:.2f}] {title}: {text[:120]}...")


Q: What is the capital of France?
  [22.66] "Departments of France": limited to preventing local policy from conflicting with national policy. The departments are further divided into commu...
  [22.41] "Le Mans": Le Mans Le Mans () is a city in France, on the Sarthe River. Traditionally the capital of the province of Maine, it is n...
  [21.49] Aorta: Windkessel effect of the great elastic arteries has important biomechanical implications. The elastic recoil helps conse...

Q: Who wrote Pride and Prejudice?
  [26.96] "Pride and Prejudice": Johnson wrote the "outrageous unconventionality" of Elizabeth Bennet was in Austen's own time very daring, especially gi...
  [24.95] "Jane Austen": "By the author of "Sense and Sensibility"" and Austen's name never appeared on her books during her lifetime. Egerton th...
  [23.95] "Pride and Prejudice": the Military Library, Whitehall in exchange for £110 (Austen had asked for £150). This proved a costly decision. Austen ...

Q: When did World War

In [ ]:
import sys
#project_dir = "/content/gdrive/MyDrive/Colab Notebooks/final-proj"
project_dir = "/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from eval_harness import load_dataset, eval_pipeline, save_results, print_summary
from rag_arch import make_colbert_retriever, make_dpr_retriever, make_bart_generator
from dataset import exact_match_score as em
print("All imports successful")

# Rebuild bart_fn with the new BART
bart_fn = make_bart_generator(
    model=bart_model,
    tokenizer=bart_tokenizer,
    device="cuda",
    max_new_tokens=128,
    num_beams=4,
)
colbert_fn = make_colbert_retriever(model=model, retriever=colbert_retriever, passages_df=passages_sample, id_to_row=None)
dpr_fn = make_dpr_retriever(faiss_index=dpr_index, passages_df=passages_sample, q_encoder=q_encoder, q_tokenizer=q_tokenizer)

# Re-test
test_q = "What is the capital of France?"
print(f"\nQ: {test_q}")
print("\n— ColBERT + BART —")
docs = colbert_fn(test_q, k=5)
print(f"  Top retrieved: {docs[0]['title']}")
print(f"  Answer: {bart_fn(test_q, docs)}")

print("\n— DPR + BART —")
docs = dpr_fn(test_q, k=5)
print(f"  Top retrieved: {docs[0]['title']}")
print(f"  Answer: {bart_fn(test_q, docs)}")


All imports successful

Q: What is the capital of France?

— ColBERT + BART —
  Top retrieved: "Departments of France"
  Answer: The capital of France is Paris. It was the capital of the Kingdom of France for a long time, but it was moved to Versailles during the French Revolution.

— DPR + BART —
  Top retrieved: Guangzhou
  Answer: The capital of France is Paris. It was the capital of the French Republic from 1789 to 1848, when it became the de jure capital. The current capital is Saint-Germain-des-Prés.


In [ ]:
#Don't run cell (optional)

import sys
project_dir = "/content/gdrive/MyDrive/Colab Notebooks/final-proj"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Force fresh imports of any updated modules
for mod in ["eval_harness", "rag_arch", "metrics", "dataset"]:
    if mod in sys.modules:
        del sys.modules[mod]

from eval_harness import load_dataset, eval_pipeline, save_results, print_summary
# Load 10 NQ dev examples
nq_dev = load_dataset(
    "/content/gdrive/MyDrive/Colab Notebooks/final-proj/data/nq_dev.jsonl",
    max_examples=10,
)

# Run ColBERT + BART
print("=" * 60)
print("Running ColBERT + BART on 10 NQ questions...")
result_colbert = eval_pipeline(colbert_fn, bart_fn, nq_dev, k=5)

print_summary(result_colbert, label="ColBERT + BART (10 NQ)")

# Show each prediction
print("\n--- Predictions ---")
for p in result_colbert["predictions"]:
    p_score = p.get("em_score", p.get("EM_score", 0))
    mark = "✓" if p_score else "✗"
    print(f"\n{mark} Q: {p['question']}")
    print(f"   Pred: {p['prediction'][:150]}")
    print(f"   Gold: {p['answers']}")


ModuleNotFoundError: No module named 'eval_harness'

In [ ]:
for q in ["when did the eagles win last super bowl", "who was the ruler of england in 1616"]:
    docs = colbert_fn(q, k=5)
    print(f"\nQ: {q}")
    for i, d in enumerate(docs):
        print(f"  [{i}] {d['title']}: {d['text'][:150]}")
    print(f"  Pred: {bart_fn(q, docs)}")


Q: when did the eagles win last super bowl
  [0] "Philadelphia Eagles": goal by Jake Elliott to make the final score 41–33. The franchise won their first Super Bowl ever and their first championship since 1960. Foles won S
  [1] "Philadelphia Eagles": and they also drafted DeSean Jackson, a receiving threat when paired with McNabb. On January 11, 2009, the team defeated the defending Super Bowl cham
  [2] "Super Bowl": their dark-colored uniform in more recent years are the Green Bay Packers against the Pittsburgh Steelers in Super Bowl XLV and the Philadelphia Eagle
  [3] "Super Bowl XV": receptions. Eagles running back Wilbert Montgomery led Philadelphia in rushing and receiving with 44 rushing yards and 6 receptions for 91 yards. The 
  [4] "Super Bowl XV": 88-yard, 12-play drive that was capped by Jaworski's 8-yard touchdown pass to tight end Keith Krepfle. But on their ensuing drive, Oakland marched fro
  Pred: The last time the Eagles won a Super Bowl was in 1960, when they beat

In [ ]:
# Load NQ dev examples for strict retrieval-recall debugging

from pathlib import Path
from eval_harness import load_dataset

PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")

if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj")

DATA_PATH = PROJECT_DIR / "data/nq_dev.jsonl"

print("DATA_PATH:", DATA_PATH)
assert DATA_PATH.exists(), f"Missing {DATA_PATH}"

# Use 500 for quick retrieval debugging.
# Change to None if you want all 3,610 examples.
nq_dev = load_dataset(str(DATA_PATH), max_examples=500)

print(f"Loaded {len(nq_dev)} NQ dev examples")
print(nq_dev[0])

DATA_PATH: /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/data/nq_dev.jsonl
Loaded 500 NQ dev examples
{'question': 'when was the last time anyone was on the moon', 'answers': ['14 December 1972 UTC', 'December 1972']}


In [ ]:
from dataset import normalize_answer
from tqdm.auto import tqdm

def good_answer_variants(answers):
    cleaned = []

    for a in answers:
        a_norm = normalize_answer(a)

        # Ignore very short ambiguous answers like "one"
        if len(a_norm) >= 4 or len(a_norm.split()) >= 2:
            cleaned.append(a_norm)

    return cleaned


def contains_good_answer(text, answers):
    text_n = normalize_answer(text)
    variants = good_answer_variants(answers)

    return any(a in text_n for a in variants)


def recall_at_k_strict(retriever_fn, examples, k=5, max_examples=500):
    hits = 0
    total = 0

    for ex in tqdm(examples[:max_examples]):
        q = ex["question"]
        answers = ex.get("answers") or ex.get("all_answers") or []

        variants = good_answer_variants(answers)
        if not variants:
            continue

        docs = retriever_fn(q, k=k)

        found = any(
            contains_good_answer(d.get("text", str(d)), answers)
            for d in docs
        )

        hits += int(found)
        total += 1

    return hits / total * 100, hits, total


colbert_recall, hits, total = recall_at_k_strict(
    colbert_fn,
    nq_dev,
    k=5,
    max_examples=500,
)

print(f"Strict ColBERT Recall@5: {colbert_recall:.2f}% ({hits}/{total})")

  0%|          | 0/500 [00:00<?, ?it/s]

Strict ColBERT Recall@5: 32.78% (159/485)


In [ ]:
dpr_recall, hits, total = recall_at_k_strict(
    dpr_fn,
    nq_dev,
    k=5,
    max_examples=500,
)

print(f"Strict DPR Recall@5: {dpr_recall:.2f}% ({hits}/{total})")

  0%|          | 0/500 [00:00<?, ?it/s]

Strict DPR Recall@5: 14.43% (70/485)


In [ ]:
# from eval_harness import load_dataset, eval_pipeline, save_results, print_summary
# from dataset import exact_match_score, squad_f1, normalize_answer

# # Load full dev set (drop max_examples, or set high)
# nq_dev = load_dataset(
#     "/content/gdrive/MyDrive/Colab Notebooks/final-proj/data/nq_dev.jsonl",
#     max_examples=None,  # or whatever your loader uses for "all"
# )
# print(f"Loaded {len(nq_dev)} examples")

# # ColBERT + BART
# print("=" * 60)
# print("ColBERT + BART")
# result_colbert = eval_pipeline(colbert_fn, bart_fn, nq_dev, k=5)
# print_summary(result_colbert, label="ColBERT + BART")
# save_results(result_colbert, "/content/gdrive/MyDrive/Colab Notebooks/final-proj/results_colbert_bart.json")

# # DPR + BART
# print("=" * 60)
# print("DPR + BART")
# result_dpr = eval_pipeline(dpr_fn, bart_fn, nq_dev, k=5)
# print_summary(result_dpr, label="DPR + BART")
# save_results(result_dpr, "results_dpr_bart.json")

from pathlib import Path
import json

from eval_harness import load_dataset, eval_pipeline, save_results, print_summary
from dataset import exact_match_score, squad_f1, normalize_answer

# -----------------------------
# Paths
# -----------------------------
PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")

if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj")

DATA_PATH = PROJECT_DIR / "data/nq_dev.jsonl"

RESULTS_DIR = PROJECT_DIR / "outputs/colbert_open_domain_eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Saving results to:", RESULTS_DIR)

# -----------------------------
# Eval config
# -----------------------------
K = 5
MAX_EXAMPLES = None   # use full NQ dev set; change to 500 for quick test

config = {
    "task": "open_domain_qa",
    "dataset": "Natural Questions dev",
    "data_path": str(DATA_PATH),
    "k": K,
    "max_examples": MAX_EXAMPLES,
    "systems": ["ColBERT+BART", "DPR+BART"],
}

with open(RESULTS_DIR / "run_config.json", "w") as f:
    json.dump(config, f, indent=2)

# -----------------------------
# Load data
# -----------------------------
nq_dev = load_dataset(str(DATA_PATH), max_examples=MAX_EXAMPLES)
print(f"Loaded {len(nq_dev)} examples")

# -----------------------------
# Helper: run or load checkpoint
# -----------------------------
def run_or_load(label, retriever_fn, generator_fn, out_name):
    out_path = RESULTS_DIR / out_name

    if out_path.exists():
        print("=" * 60)
        print(f"{label} already exists. Skipping rerun.")
        print("Saved file:", out_path)
        with open(out_path, "r") as f:
            result = json.load(f)
        print_summary(result, label=label)
        return result

    print("=" * 60)
    print(label)

    result = eval_pipeline(
        retriever_fn,
        generator_fn,
        nq_dev,
        k=K,
    )

    print_summary(result, label=label)
    save_results(result, str(out_path))

    print("Saved:", out_path)
    return result

# -----------------------------
# Run evaluations
# -----------------------------
result_colbert = run_or_load(
    label="ColBERT + BART",
    retriever_fn=colbert_fn,
    generator_fn=bart_fn,
    out_name=f"results_colbert_bart_k{K}.json",
)

result_dpr = run_or_load(
    label="DPR + BART",
    retriever_fn=dpr_fn,
    generator_fn=bart_fn,
    out_name=f"results_dpr_bart_k{K}.json",
)

# -----------------------------
# Save compact comparison summary
# -----------------------------
summary = {
    "k": K,
    "num_examples": len(nq_dev),
    "colbert_result_file": str(RESULTS_DIR / f"results_colbert_bart_k{K}.json"),
    "dpr_result_file": str(RESULTS_DIR / f"results_dpr_bart_k{K}.json"),
    "colbert": result_colbert,
    "dpr": result_dpr,
}

summary_path = RESULTS_DIR / f"summary_k{K}.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print("Final comparison saved to:", summary_path)


Saving results to: /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/outputs/colbert_open_domain_eval
Loaded 3610 examples
ColBERT + BART


Evaluating...: 100%|██████████| 3610/3610 [1:22:51<00:00,  1.38s/it]



  ColBERT + BART
  EM Score:    0.00
  F1 Score:    0.02
  Examples: 3,610
  Time:     4971.8s (1.38s/ex)
Saved: /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/outputs/colbert_open_domain_eval/results_colbert_bart_k5.json
DPR + BART


Evaluating...:   5%|▌         | 187/3610 [04:17<1:18:25,  1.37s/it]


KeyboardInterrupt: 

In [ ]:
# Use the fine-tuned NQ RAG-Sequence BART generator.
# This keeps BART, but replaces only the retriever with ColBERT.

import torch
from pathlib import Path
from transformers import RagSequenceForGeneration, RagTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Try to find your trained NQ RAG-Sequence checkpoint.
NQ_CKPT_CANDIDATES = [
    Path("outputs/rag_baseline_nq/best"),
    Path("outputs/rag_baseline_nq/final"),
    Path("outputs/rag_baseline_nq/checkpoint-2000"),
    Path("outputs/rag_baseline_nq/checkpoint-1000"),
]

NQ_CKPT = None
for p in NQ_CKPT_CANDIDATES:
    if p.exists():
        NQ_CKPT = p
        break

if NQ_CKPT is None:
    raise FileNotFoundError("Could not find trained NQ checkpoint. Check outputs/rag_baseline_nq/")

print("Loading fine-tuned NQ RAG checkpoint:", NQ_CKPT)

# Load the trained RAG model, but we will only use its BART generator.
rag_model = RagSequenceForGeneration.from_pretrained(
    str(NQ_CKPT),
    retriever=None,
).to(DEVICE)

rag_model.eval()

rag_tokenizer = RagTokenizer.from_pretrained(str(NQ_CKPT))

# This is the BART generator fine-tuned during your NQ RAG training.
nq_bart_model = rag_model.rag.generator.to(DEVICE).eval()
nq_bart_tokenizer = rag_tokenizer.generator

def format_rag_context(question, doc):
    title = str(doc.get("title", ""))
    text = str(doc.get("text", str(doc)))
    return f"{question} // {title} // {text[:1200]}"

@torch.no_grad()
def bart_fn(question, docs, max_new_tokens=16, num_beams=4):
    """
    Fine-tuned BART generator from NQ RAG-Sequence.
    Uses ColBERT-retrieved docs as context.
    Returns short NQ-style answers.
    """
    if not docs:
        return ""

    contexts = [format_rag_context(question, d) for d in docs[:5]]

    enc = nq_bart_tokenizer(
        contexts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    outputs = nq_bart_model.generate(
        **enc,
        num_beams=num_beams,
        num_return_sequences=1,
        max_new_tokens=max_new_tokens,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

    preds = nq_bart_tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Return the first non-empty answer, ordered by ColBERT rank.
    for p in preds:
        p = p.strip()
        if p:
            return p

    return ""

print("Replaced bart_fn with fine-tuned NQ RAG BART generator.")

Loading fine-tuned NQ RAG checkpoint: outputs/rag_baseline_nq/best


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie rag.generator.model.shared.weight to rag.generator.model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie rag.generator.model.shared.weight to rag.generator.model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Replaced bart_fn with fine-tuned NQ RAG BART generator.


In [ ]:
#smoke test
q = "when was the last time anyone was on the moon"
docs = colbert_fn(q, k=5)

print("Question:", q)
print("Top doc:", docs[0]["title"])
print("Generated:", bart_fn(q, docs))

Question: when was the last time anyone was on the moon
Top doc: NQ train example 40934
Generated: 11 December 1972


In [ ]:
# Recall@K sweep — run after colbert_fn and nq_dev exist

for K in [5, 10, 20, 50]:
    colbert_recall, hits, total = recall_at_k_strict(
        colbert_fn,
        nq_dev,
        k=K,
        max_examples=500,
    )
    print(f"Strict ColBERT Recall@{K}: {colbert_recall:.2f}% ({hits}/{total})")

  0%|          | 0/500 [00:00<?, ?it/s]

Strict ColBERT Recall@5: 32.78% (159/485)


  0%|          | 0/500 [00:00<?, ?it/s]

Strict ColBERT Recall@10: 35.67% (173/485)


  0%|          | 0/500 [00:00<?, ?it/s]

Strict ColBERT Recall@20: 38.76% (188/485)


  0%|          | 0/500 [00:00<?, ?it/s]

Strict ColBERT Recall@50: 44.33% (215/485)


In [ ]:
#new eval cell after debugging (only 500 to check if worth scaling)

from pathlib import Path
import json

from eval_harness import load_dataset, eval_pipeline, save_results, print_summary

PROJECT_DIR = Path("/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj")

if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj")

DATA_PATH = PROJECT_DIR / "data/nq_dev.jsonl"

#RESULTS_DIR = PROJECT_DIR / "outputs/colbert_open_domain_eval_finetuned_bart"
#RESULTS_DIR = PROJECT_DIR / "outputs/colbert_open_domain_eval_adapted_bart_5k"
RESULTS_DIR = PROJECT_DIR / "outputs/colbert_open_domain_eval_adapted_bart_5k_k50"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Saving results to:", RESULTS_DIR)

K = 50
MAX_EXAMPLES = 500

config = {
    "task": "open_domain_qa",
    "dataset": "Natural Questions dev",
    "retriever": "ColBERT over data/small_nq_index/passages",
    "generator": "BART",
    "data_path": str(DATA_PATH),
    "k": K,
    "max_examples": MAX_EXAMPLES,
}

with open(RESULTS_DIR / "run_config.json", "w") as f:
    json.dump(config, f, indent=2)

nq_dev_eval = load_dataset(str(DATA_PATH), max_examples=MAX_EXAMPLES)
print(f"Loaded {len(nq_dev_eval)} examples")

print("=" * 60)
print("ColBERT + BART over small_nq_index corpus")

result_colbert = eval_pipeline(
    colbert_fn,
    bart_fn,
    nq_dev_eval,
    k=K,
)

print_summary(result_colbert, label="ColBERT + BART small_nq_index")

out_path = RESULTS_DIR / f"results_colbert_bart_small_nq_k{K}_{len(nq_dev_eval)}.json"
save_results(result_colbert, str(out_path))

print("Saved:", out_path)

Saving results to: /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/outputs/colbert_open_domain_eval_adapted_bart_5k_k50
Loaded 500 examples
ColBERT + BART over small_nq_index corpus


Evaluating...: 100%|██████████| 500/500 [01:17<00:00,  6.42it/s]



  ColBERT + BART small_nq_index
  EM Score:    0.18
  F1 Score:    0.23
  Examples: 500
  Time:     77.9s (0.16s/ex)
Saved: /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/outputs/colbert_open_domain_eval_adapted_bart_5k_k50/results_colbert_bart_small_nq_k50_500.json
